# Architecture Ablations — ML-1M / LightGCN (standalone)

The two ablations Reviewer 1 **requires** for the camera-ready (point 3):

1. **AIE: attention vs fixed propagation.** The direct test of the paper's
   central question. `--aie_fixed_prop` swaps every `GATInfluenceLayer` for a
   `FixedPropInfluenceLayer`: identical linear transform W, identical residual
   ELU stacking, identical mean readout and shift generator — but influence
   propagates with the FIXED degree-normalized coefficients of Ā_Δ instead of
   learned attention α_ij. Attention is the only variable.
2. **GAIE: variational vs deterministic autoencoder.** `--gaie_deterministic`
   keeps the same encoder and reconstruction objective but uses Z = μ always
   (no sampling) and drops the KL prior. Answers "does the variational part
   matter", which the ±0.002 effect of removing L_rec alone left open.

**Fully standalone.** This notebook trains its own backbone and runs all
**four** variants itself — full AIE, fixed-prop AIE, full GAIE, deterministic
GAIE — so every pairwise comparison is same-session, same-device,
same-hyperparameters by construction. It needs nothing from the other
notebooks and nothing uploaded.

Scope: ML-1M / LightGCN, seed 1234. ~4 h on a Kaggle T4 (4 pipelines ≈ 50 min
each, plus ~20 min backbone). Results → `./logs/ablation_arch_ml1m_lightgcn.json`,
written incrementally (re-running skips completed variants).

### Protocol switch
`HELD_OUT = False` (default) uses the original actual-request selection,
matching the current main table. Set `HELD_OUT = True` to run everything under
the leakage-free held-out protocol instead. Either way all four variants share
one protocol in one session, so the pairwise deltas are always clean.

### Kaggle setup
Settings → Accelerator → **GPU**. That's all.

Requires the repo at a commit with `--aie_fixed_prop` / `--gaie_deterministic`.

---
## 0. Environment

In [ ]:
import os
import subprocess
import torch
cuda_tag = torch.version.cuda.replace(".", "")        # e.g. "121" or "124"
torch_tag = ".".join(torch.__version__.split(".")[:2]) # e.g. "2.6"
whl_url = f"https://data.pyg.org/whl/torch-{torch_tag}.0+cu{cuda_tag}.html"
print(f"Installing torch-scatter + torch-sparse from: {whl_url}")
subprocess.check_call(["pip", "install", "-q", "torch-scatter", "torch-sparse", "-f", whl_url])
subprocess.check_call(["pip", "install", "-q", "setproctitle"])

import torch_scatter
import torch_sparse
import setproctitle
print("torch_scatter version:", torch_scatter.__version__)
print("torch_sparse version:", torch_sparse.__version__)
print("setproctitle installed \u2713")

In [ ]:
import os

REPO_URL = "https://github.com/Shuvayu12/unlearnrec_improv.git"
PROJECT_DIR = "/kaggle/working/unlearnrec_improv"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already cloned.")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
import sys

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Clear sys.argv so argparse in config/params.py doesn't choke on notebook kernel args
sys.argv = [sys.argv[0]]

from config.params import args
from data.data_handler import DataHandler
from Utils.time_logger import log
from models.Model import LightGCN, AIE, GAIE
from training.pretrain_simgcl import Coach as PretrainCoach   # dispatches on args.model
from unlearning.aie_unlearn import Coach as AIECoach
from unlearning.gaie_unlearn import Coach as GAIECoach

import torch as t
import numpy as np
import random
import time
import json
import glob
import traceback

# Fail fast if the repo commit lacks the ablation flags
assert hasattr(args, 'aie_fixed_prop') and hasattr(args, 'gaie_deterministic'), (
    'This repo clone has no ablation support; pull the commit that adds '
    '--aie_fixed_prop / --gaie_deterministic to config/params.py and models/Model.py.')

os.makedirs("./ckpt", exist_ok=True)
os.makedirs("./logs", exist_ok=True)

print(f"CUDA available: {t.cuda.is_available()}")
if t.cuda.is_available():
    print(f"GPU: {t.cuda.get_device_name(0)}")

---
## 1. Hyperparameters (inline — zero setup)

Identical to the other two notebooks (effective ml1m/lightgcn configuration of
the committed 3x3 notebooks). Identical by design: the ablated component must
be the only variable against the baseline rows.

In [ ]:
HP = {
    'unlearn':       dict(epoch=30, batch=4096, pretrain_drop_rate=0.1,
                          sim_epoch=10, unlearn_wei=1.0, align_wei=0.01,
                          align_temp=1.0, perf_degrade=0.3),
    'gaie_extra':    dict(rec_wei=0.03, perf_degrade=0.5),
    'finetune_aie':  dict(epoch=20, unlearn_wei=0.2, align_wei=0.01),
    'finetune_gaie': dict(epoch=15, unlearn_wei=0.2, align_wei=0.01),
    'backbone':      dict(pretrain_epoch=50, batch=2048, reg=1e-7,
                          adv_method='lightgcn0.5'),
}
HP_SOURCE = 'inline notebook defaults (ml1m/lightgcn, matches the 3x3 notebooks)'

# Optional override -- no setup required (HPARAMS_PATH env var or hparams.json
# in an attached Kaggle input dataset; non-null keys replace the defaults).
_cand = os.environ.get('HPARAMS_PATH')
if not (_cand and os.path.exists(_cand)):
    _hits = sorted(glob.glob('/kaggle/input/*/hparams.json'))
    _cand = _hits[0] if _hits else None
if _cand:
    with open(_cand) as fs:
        _ext = json.load(fs)
    for _sec, _vals in _ext.items():
        HP.setdefault(_sec, {}).update(
            {k: v for k, v in _vals.items() if v is not None})
    HP_SOURCE = f'inline defaults overridden by {_cand}'

print('Hyperparameters:', HP_SOURCE)

---
## 2. Configuration & harness

In [ ]:
DATASET = 'ml1m'
BACKBONE = 'lightgcn'
SEED = 1234
FT_TUNE_E0 = True

# False: original actual-request selection (matches the current main table).
# True: leakage-free held-out selection (matches the held-out notebook).
# All four variants run under the chosen protocol in this session.
HELD_OUT = False
SELECTION_SEED = 5678

RESULTS_FILE = './logs/ablation_arch_ml1m_lightgcn.json'
all_results = {}
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as fs:
        all_results = json.load(fs).get('results', {})
    print(f'Resuming: {len(all_results)} entries already in {RESULTS_FILE}')


def save_results():
    with open(RESULTS_FILE, 'w') as fs:
        json.dump({'dataset': DATASET, 'backbone': BACKBONE, 'seed': SEED,
                   'held_out_selection': HELD_OUT, 'ft_tune_e0': FT_TUNE_E0,
                   'protocol': ('held-out simulated selection' if HELD_OUT
                                else 'actual-request selection (original)'),
                   'results': all_results}, fs, indent=2)


def reset_seeds(seed):
    t.manual_seed(seed)
    t.cuda.manual_seed_all(seed)
    t.backends.cudnn.deterministic = True
    np.random.seed(seed)
    random.seed(seed)


def apply_args(d):
    for k, v in d.items():
        setattr(args, k, v)


def set_common(seed=SEED):
    args.data = DATASET
    args.model = BACKBONE
    args.seed = seed
    args.gpu = '0'
    args.lr = 1e-3
    args.latdim = 128
    args.gnn_layer = 3
    args.topk = 20
    args.tst_bat = 256
    args.decay = 1.0
    args.bpr_wei = 1.0
    args.load_model = None
    args.reg = HP['backbone']['reg']
    args.adv_method = HP['backbone']['adv_method']
    args.adversarial_attack = True
    # ablation flags default OFF; each variant sets its own
    args.aie_fixed_prop = False
    args.gaie_deterministic = False
    # selection protocol
    args.held_out_selection = HELD_OUT
    args.selection_seed = SELECTION_SEED
    args.exclude_adv_from_sim = HELD_OUT


UNLEARN_STRUCTURAL = dict(
    lr=1e-3, test_drop_rate=0.003, tst_epoch=5, bpr_wei=1.0,
    align_type='v2', unlearn_type='v1', overall_withdraw_rate=0.1,
    withdraw_rate_init=1, hyper_temp=1.0, unlearn_ssl=1e-3,
    layer_mlp=2, leaky=0.99, act='leaky', unlearn_layer=0,
)


def unlearn_cfg(pipeline):
    cfg = dict(UNLEARN_STRUCTURAL)
    u = HP['unlearn']
    cfg.update(epoch=u['epoch'], batch=u['batch'],
               pretrain_drop_rate=u['pretrain_drop_rate'], sim_epoch=u['sim_epoch'],
               unlearn_wei=u['unlearn_wei'], align_wei=u['align_wei'],
               align_temp=u['align_temp'], perf_degrade=u['perf_degrade'])
    if pipeline == 'gaie':
        g = HP['gaie_extra']
        cfg['rec_wei'] = g['rec_wei']
        cfg['perf_degrade'] = g.get('perf_degrade', cfg['perf_degrade'])
    return cfg


print(f'Config: {DATASET}/{BACKBONE}, seed={SEED}, HELD_OUT={HELD_OUT}')

---
## 3. Attacked backbone (shared with the other notebooks)

In [ ]:
set_common()
args.epoch = HP['backbone']['pretrain_epoch']
args.batch = HP['backbone']['batch']
args.tst_epoch = 10
BACKBONE_CKPT = f'./ckpt/pre_{DATASET}_{BACKBONE}_adv'
args.save_path = BACKBONE_CKPT
reset_seeds(SEED)

handler_atk = DataHandler()
handler_atk.load_data(drop_rate=0.0, adv_attack=True)
print(f'users={args.user}, items={args.item}, '
      f'injected adversarial edges={len(handler_atk.adv_edges[0])}')

if not os.path.exists(BACKBONE_CKPT + '.mod'):
    _t0 = time.time()
    PretrainCoach(handler_atk).run()
    print(f'Attacked backbone trained in {time.time() - _t0:.1f}s')
else:
    print(f'Reusing checkpoint {BACKBONE_CKPT}.mod')

_m = t.load(BACKBONE_CKPT + '.mod', weights_only=False)['model'].cuda()
_m.training = False
BACKBONE_RES = dict(PretrainCoach(handler_atk).tst_epoch(_m))
print(f'[Attacked backbone] Recall@{args.topk}={BACKBONE_RES["Recall"]:.4f}, '
      f'NDCG@{args.topk}={BACKBONE_RES["NDCG"]:.4f}')
del _m, handler_atk
t.cuda.empty_cache()

---
## 4. Ablation runs — all four variants, this session

| pipeline | variant | flag |
|---|---|---|
| AIE | `attention` (full) | — |
| AIE | `fixed_prop` | `aie_fixed_prop=True` |
| GAIE | `variational` (full) | — |
| GAIE | `deterministic` | `gaie_deterministic=True` |

The two `full` baselines are rerun here on purpose: hyperparameters may differ
across sessions/devices, so the only trustworthy baseline is the one produced
in the same session as its ablated counterpart.

Note on the deterministic GAIE: the log-variance head still exists in the
module but receives no gradient (Z = μ, no KL), so parameter counts stay
comparable; only the stochastic/variational mechanism is removed.

In [ ]:
COACHES = {'aie': AIECoach, 'gaie': GAIECoach}


def run_variant(pipeline, variant, flags):
    coach_cls = COACHES[pipeline]

    # ---- stage 1: encoder pre-training ----
    set_common()
    apply_args(unlearn_cfg(pipeline))
    apply_args(flags)           # ablation flag BEFORE the model is built
    args.fineTune = False
    args.trained_model = BACKBONE_CKPT
    args.save_path = f'./ckpt/abl_{pipeline}_{variant}'
    reset_seeds(SEED)
    h_u = DataHandler()
    if HELD_OUT:
        h_u.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=False, exclude_adv=True)
    else:
        h_u.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)
    coach = coach_cls(h_u)
    _t0 = time.time()
    coach.run()
    t_unlearn = time.time() - _t0

    # ---- stage 2: fine-tuning (E0 unfrozen) ----
    args.fineTune = True
    args.ft_tune_e0 = FT_TUNE_E0
    args.model_2_finetune = args.save_path
    ft = HP[f'finetune_{pipeline}']
    apply_args(dict(epoch=ft['epoch'], unlearn_wei=ft['unlearn_wei'],
                    align_wei=ft['align_wei']))
    args.save_path = args.save_path + '_ft'
    reset_seeds(SEED)
    h_f = DataHandler()
    if HELD_OUT:
        # closes the negative-stloc leak (see held-out notebook, Section 4)
        h_f.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=False, exclude_adv=True)
    else:
        # original protocol
        h_f.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)
    coach = coach_cls(h_f)
    _t0 = time.time()
    coach.run()
    t_ft = time.time() - _t0

    # ---- evaluation: best checkpoint, actual request, deterministic ----
    h_e = DataHandler()
    h_e.load_data(drop_rate=args.test_drop_rate, adv_attack=True)
    sp = args.save_path + '.mod'
    if os.path.exists(sp):
        coach.model = t.load(sp, weights_only=False)['model'].cuda()
    coach.handler = h_e
    coach.model.eval()      # GAIE: Z = mu at inference (paper Sec 3.2)
    reset_seeds(SEED)       # reproducible evaluation negatives
    metrics = coach.tst_epoch(coach.model)
    mi = coach.test_unlearn(coach.model, prefix=f'[{pipeline.upper()}/{variant}] final')

    row = dict(variant=variant, flags=flags,
               Recall=metrics['Recall'], NDCG=metrics['NDCG'],
               MI_BF=mi['mi_bf'], MI_NG=mi['mi_ng'],
               MI_AUC=mi['mi_auc'], MI_ACC=mi['mi_acc'],
               BeforeProb=mi['avg_before_prob'], AfterProb=mi['avg_after_prob'],
               NegProb=mi['avg_neg_prob'],
               UnlearnTime=t_unlearn, FinetuneTime=t_ft,
               BackboneRecall=BACKBONE_RES['Recall'], BackboneNDCG=BACKBONE_RES['NDCG'])
    del coach, h_u, h_f, h_e
    t.cuda.empty_cache()
    return row


# All four variants, always — the baselines are produced in this same session.
VARIANTS = [
    ('aie',  'attention',     {}),
    ('aie',  'fixed_prop',    {'aie_fixed_prop': True}),
    ('gaie', 'variational',   {}),
    ('gaie', 'deterministic', {'gaie_deterministic': True}),
]
print('Variants to run:', [(p, v) for p, v, _ in VARIANTS])

for pipeline, variant, flags in VARIANTS:
    key = f'{pipeline}/{variant}'
    if key in all_results and 'error' not in all_results[key]:
        print(f'skip {key} (already done)')
        continue
    print('\n' + '#' * 100)
    print(f'### {pipeline.upper()}/{variant} — {DATASET}/{BACKBONE} seed {SEED}')
    print('#' * 100)
    try:
        all_results[key] = run_variant(pipeline, variant, flags)
    except Exception:
        traceback.print_exc()
        all_results[key] = {'error': traceback.format_exc()[-1500:]}
    save_results()
    print(f'saved {RESULTS_FILE}')

---
## 5. Ablation tables

Two pairwise comparisons, each isolating one architectural choice. All rows
come from this session, so each pair shares its backbone, hyperparameters,
device, and selection protocol by construction.

In [ ]:
METRICS = ['Recall', 'NDCG', 'MI_BF', 'MI_NG', 'MI_AUC', 'MI_ACC']


def get_row(pipeline, variant):
    r = all_results.get(f'{pipeline}/{variant}')
    return r if (r and 'error' not in r) else None


PAIRS = [
    ('aie',  [('attention',  'learned attention (full AIE)'),
              ('fixed_prop', 'fixed degree-normalized propagation')]),
    ('gaie', [('variational',   'variational (full GAIE)'),
              ('deterministic', 'deterministic autoencoder (Z=μ, no KL)')]),
]

for pipeline, variants in PAIRS:
    print(f'\n=== {pipeline.upper()} — {DATASET}/{BACKBONE}, seed {SEED} ===')
    hdr = f'{"variant":<42}' + ''.join(f'{m:>9}' for m in METRICS)
    print(hdr)
    print('-' * len(hdr))
    rows = {}
    for v, label in variants:
        r = get_row(pipeline, v)
        rows[v] = r
        if r is None:
            print(f'{label:<42}{"(missing)":>9}')
        else:
            print(f'{label:<42}' + ''.join(f'{r[m]:>9.4f}' for m in METRICS))
    base_v, abl_v = variants[0][0], variants[1][0]
    if rows.get(base_v) and rows.get(abl_v):
        print(f'{"delta (ablated - full)":<42}' +
              ''.join(f'{rows[abl_v][m] - rows[base_v][m]:>+9.4f}' for m in METRICS))

print('\n% ---- LaTeX (architecture ablations) ----')
print(r'\begin{tabular}{llcccc}')
print(r'\toprule')
print(r'Encoder & Variant & Recall@20 & NDCG@20 & MI-BF & MI-NG \\')
print(r'\midrule')
for pipeline, variants in PAIRS:
    for v, label in variants:
        r = get_row(pipeline, v)
        if r:
            print(f'{pipeline.upper()} & {label} & {r["Recall"]:.4f} & '
                  f'{r["NDCG"]:.4f} & {r["MI_BF"]:.4f} & {r["MI_NG"]:.4f} \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')

---
## Notes for the paper

- **AIE pair:** if attention beats fixed propagation on MI-NG at comparable
  Recall, that is the first direct evidence for the paper's central claim —
  state it with these numbers. If it doesn't, the honest framing is that the
  gains come from the learned encoder + fine-tuning pipeline rather than the
  attention mechanism per se.
- **GAIE pair:** if variational ≈ deterministic, say plainly that the
  variational machinery is not the active ingredient (reviewer already
  suspects this from the L_rec ablation); GAIE's value is then its
  conservative utility-preserving behavior. If variational wins, this table
  is the justification the current draft lacks.
- Both comparisons hold everything fixed except one component: same losses,
  schedules, seeds, backbone checkpoint, and selection protocol as the
  baseline rows.
- The fixed-propagation layer keeps the linear transform W, so parameter
  counts differ only by the attention vector a (2d per layer) — negligible;
  the deterministic GAIE keeps the (unused, ungradiented) log-variance head.